IMPORTANTE: la matrice di covarianza dà il QUADRATO dell'incertezza sulle diagonali  
per avere l'incertezza sui parametri, devo prendere la radice  

### Funzione modello

**Quali sono i parametri da fittare?**

tutti quelli che non ho misurato direttamente, ma per cui ho solo una stima il valore nominale  

esempio:  
1- C dei condensatori  
so solo C nominale, non so come si misura  
se nella formula appare $\tau$, posso stimare direttamente quello, da cui poi trovo C  

2- $r_d$, resistenza dinamica della giunzione pn  
so solo stimare il valore atteso, ma non misurarlo  

**Tips per scrivere funzione**

1- se la funzione non fitta, aggiungi un coefficiente o un offset   
perchè: nella realtà ho vari problemi, tipo l'impedenza del generatore e dell'oscilloscopio non sono nulle  

2- se i miei punti sono f, non $\omega$, allora scrivi G in funzione di Omega  

3- spesso alcuni parametri sono collegati, quindi non fittarli separati   

**Tips 1: esempio**

esempio: aggiungi coeff A:  
$G(f, A)=\frac{A}{\sqrt{1+(f/f_c)^2}}$  

esempio: agigungi offset B:    
$V= A \,e^{-t/ \tau} +B$   

**Tips 3: esempio**

1- R e C sono collegati  
$G= \frac{ 2 \pi RC \; f}{\sqrt{(1-(f/f_0)^2)^2 +(2\pi R_{tot}C \; f)^2}} = \frac{A\; f}{\sqrt{(1-(f/f_0)^2)^2 +(B \; f)^2}}$    
so params are: A, B, f0  

2- $\beta_f$ e $R_b'$ sono correlati, perchè nella loro stima appare in entrambi $I_C$    
$T=-\frac{R_c}{R'_b + \frac{\beta+1}{Z_E}}\beta_f \sim   -\frac{R_c}{\beta(\frac{R_b'}{\beta} +Z_E)} = -\frac{R_C}{\frac{R_b'}{\beta} +Z_E}$   
so params are: $C_E$, $R_b' / \beta$ 

### Prendere valori per fit

. | Time domain | Freq domain
---|---|---
cosa misuro | con programma per computer, prendi input V e t  |a mano nell'oscilloscopio, misura V per diversi valori della freq del generatore  
cosa plottare | semplicemente V(t) | un bode plot  
se voglio solo errore statistico | durante l'acquisizione da computer, prendi output in digit | misura tutti i valori nell'oscilloscopio sula stessa scala (in teoria anche lo stesso canale, ma vabbé)  
se voglio stddev | dì anche al programma di fare la media su più sweep | dividi l'errore statistico per $\sqrt{12}$ (assumo distribuzione uniforme dell'errore)

**Che punti prendere**

1- dove li voglio  
immagina come sarà la curva  
mi servono molti punti nella parte più particolare, quindi non nella parte piatta  
tipo, in un filtro, tanti vicino alla freq di taglio  
o in carica condensatore, tanto nella parte di carica/scarica, ma pochi nella parte asintotica  

2- come sono distanziati  
se plotto gain, prendi frequenze distanziate logaritmicamente, così i punti nel grafico verrano belli  

3- da dove parte  
se time domain, l'array dei tempi deve partire da 0, sennò non fitta  

**Programma per acquisizione: output in volt o digit?**

dipende dal parametro che voglio stimare:  
se dipende solo dal tempo, allora posso prendere output in digit  
se posso prendere output in digit, fallo, mi toglie errore di calibrazione  

esempi: voglio stimare  
$\tau$: dipende solo dal tempo, prendo in digit  
$C$: NON dipende dal tempo, prendi in volt   

quando in digit: circuiti RC o RLC  
cioè, quando studio in time domain???  

**Per fare rampa**

per farla partire da V=0 a t=0:  
1- ruota la manopola per spostare orizzontalmente lo schermo finchè è tutto a sinistra  
2- metti il livello del trigger a V=0  

prima di prendere i punti dal computer, zoomma più che puoi la figura, soprattutto in altezza  
sennò non sto sfruttando la sensibilità  

### Errore 

**Differenza**

. | err statistico | err sistematico 
---|---|---
come si vede nel grafico | valore assoluto costante tipo $\pm 1 \; digit$ | valore relativo costante, tipo $\pm 5\% \; lettura$  
absolute_sigma | True | False 
test chi quadro | sì, ma devo trasformalo in stddev | no, è un kappa quadro

#### Cancella errore sistematico: time domain

**Cosa è errore di calibrazione**

l'errore di calibrazione è un errore RELATIVO COSTANTE  
cioè: $V \rightarrow V(1+\epsilon)$  
$\epsilon$ è quanto vale l'errore, tipo 3%  
l'unica cosa che fa la calibrazione è cambiare è scaling (stretch) la scala verticale (rescales all data by the same factor)  

**Quando posso cancellarlo**

per vedere se lo posso ignorare, pluggalo nella formula, vedi se la cosa che voglio stimare cambia o no  
in un rapporto, tipo il gain, gli $(1+\epsilon)$ si possono cancellare solo se gli $\epsilon$ valgono uguale  

**Esempio**

$V(t)=V_0 e^{-t / \tau}$  

errore di calibrazione cambia solo:  
$V_0(1+\epsilon)e^{-t / \tau}$  
che non cambia $\tau$, ciò che voglio trovare  
quindi posso ignorare l'errore di calibrazione, passo come errore e plotto solo quello di lettura  

spiegazione grafica:  
in semilog, è una retta  
con pendenza $-1/\tau$  
moltiplicare V per qualcosa cambia l'intercetta, ma non la pendenza  

#### Cancella errore sistematico: freq domain

**Quale scegliere**

metodo di calibrazione mi permette di fare il test del chi quadro, che è bene  
però devo essere in grado di prendere bene le misure, se l'onda è così piccola che non riesco a prendere i punti precisi non ha senso  

**Metodi per fare misure**

metodi per fare misure di due voltaggi diversi, tra cui voglio calcoalre il gain  

. | Protocollo | Calibrazione 
---|---|---
come | uso sempre la VOLT/DIV migliore | uso sempre stessa VOLT/DIV, e possibilmente anche stesso canale per le due onde  
errore | domina errore calibrazioe | domina errore statistico (no errore calibrazione)  
conseguenze | l'errore statistico è trascurabile, metto semplicemente errore di calibrazione +5% nel codice python | metto errore di lettura, e ricorda diviso $\sqrt{12}$ per avere stddev  
cosa fare dopo | absolute sigma =false, no test chi quadro | absolute sigma=True, sì test chi quadro  

### Arduino

#### Prendere in input

**Comandi**

1- boccole:  
blu: pin7, VCC  
nera: GND  
verde: TTL (time to live), per triggerare  

2- programma:   
Ssi chiama ardp, nella cartella Arduini  
per prendere solo la scarica, posso o cambiare i delay tra i campionamenti, o cambiare la frequenza dell'onda

**Come usare**

1- collega arduino al computer tramite cavo usb  

2- carica sketch (codice)  
dovrebbe essere già fatto, ma se serve:  
apri interfaccia di arduino (con simbolo di infinito)  
lo sketch è nella directory 'ARDUINI', al plurale, poi ARDP2, sketch ha estensione .ini  
aprilo e premi il pulsante upload con una freccia verso destra in un cerchio in alto a sinistra  
comunicazione tra arduino e computer avviene tramite protocollo seriale emulato su USB  

3- lancia Pyzo, usa il file ardp2.py (con estensione py, non ini come prima)  

4- run the script  
appare una finestra  
per salvare il file, cambiagli nome  
file finsice nella dir /dati_arduino 

5- per triggerare  
va messo in modalità sincrona (selezionala su ardp, c'è un pulsante)   
usa la boccola verde di Arduino, mandaci la TTL (che nell'oscilloscopio uso per triggerare)  
devo scegliere tra slope positiva e negativa (sono opzioni nel drop-down menu), così prendo carica e scarica 

6- poi il bottone a destra di quello per fare roba sincrona mi permette di fare la media (prendere tanti segnali e fare la media su tutti)  
e seleziona la casella che mi mostrale barre di errore (se ho scelto di fare la media)  

#### Calibrazione

**Calibrazione alternativa arduino**

1- collega arduino (porta 7 e GND) al generatore di onda in DC  
2- con lo script python, ottengo $N$, il valore del voltaggio misurato da arduino in [digit]  
3- intanto, col multimetro, misuro la tensione tra le due boccole di arduino che ho usato  
ottengo $V$ (Vpin7, quella chiesta dalla domanda)    
4- ho $ξ = \frac{V}{N}$  
ricorda di propagare l'errore 

**Altro metodo, più complesso**

invece di una sola coppia di punti, prendinedi più, variando la V dell'onda del generatore  
poi fittali:  
opzione 1: retta $V_v=a+bV_{dig}$  
opzione 2: forma quadratica $V_v = a+bV_{dig}+cV_{dig}^2$  
di solito parte qudratica è trascurabile, quindi $c \sim 0$  
V_dig è N, quello dato da Arduino  

### ALTRO: Potenziometro

**Collegamento**

pin esterni del potenziometro: VCC e GND (scambiabili)  
pin centrale: ???  
  
giusto???  
il - e + del generatore vanno alle boccole ESTERNE del potenziometro  
multimetro è in parallelo (quindi nei buchi verticali) tra il centrale e il GND del potenziometro  

### Cerca file

In [ ]:
# good code to serach
find $(pwd) -type f -name '*_story_8*'